<!--
"Open in Colab" badge (currently disabled).
Once this notebook is in a public GitHub repository, uncomment the badge line below and
replace <user>, <repo>, <branch> and <path-to-notebook> with the actual values, e.g.
https://colab.research.google.com/github/<user>/<repo>/blob/main/PSO_inverse_design.ipynb
Clicking the badge opens the notebook directly in Google Colab.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<user>/<repo>/blob/<branch>/<path-to-notebook>/PSO_inverse_design.ipynb)
-->

# Uncertainty-guided inverse design of a coaxial microdroplet generator with PSO

Companion notebook for the paper

> D. G. Harispe, M. Gerard, P. A. Kler. *Microdroplet Generator Surrogate Enhancement and Inverse Design Through an Uncertainty-Guided Framework.* arXiv:2611.06464 (2026). <https://arxiv.org/abs/2611.06464>

The notebook finds operating and design parameters of a **coaxial microdroplet generator** that produce a droplet of a prescribed diameter. It combines:

* two **ensembles of neural-network surrogates** trained on Volume-of-Fluid (Basilisk) simulations: a regime classifier $\mathcal{E}_r$ (dripping / narrowing jet / widening jet) and a droplet-diameter regressor $\mathcal{E}_d$, each with 20 members (Section 4 of the paper);
* a **multi-criteria, uncertainty-aware fitness function** that rewards accuracy on the target diameter, confident and unanimous prediction of the dripping regime, low disagreement among the diameter models and distance from the edges of the training domain (Section 5.2);
* a **global-best Particle Swarm Optimization (PSO)** algorithm that minimizes that fitness (Sections 3.3 and 5.1).

The notebook is self-contained. The only external resources are the trained ensemble weights, which are read from a configurable location (a local folder or a URL).

## How to use this notebook

1. **Open it in Google Colab** (or any Jupyter environment with Python ≥ 3.9). A GPU is not needed: the surrogates are small MLPs and the whole optimization runs in seconds on a CPU.
2. **Set where the trained models are** in the *Configuration* section (`MODELS_SOURCE`). This can be
   * a local folder that contains the `models/` folder (for example, after uploading it to the Colab session or mounting Google Drive),
   * a base URL that serves the same folder tree (for example, the *raw* URL of a GitHub repository), or
   * the URL or path of a `.zip` file with that folder tree.
3. **Define the design problem** in the *Configuration* section: target diameter, design variables, fixed parameters and search bounds.
4. **Optionally tune the PSO** hyperparameters and the fitness-function constants. The defaults are the values used in the paper.
5. Run all cells (*Runtime → Run all*). Section 8 runs one optimization and plots its convergence. Section 9 repeats the search with several independent swarms, as done for the validation in Section 5.3 of the paper, and exports the candidate designs to CSV. Section 10 produces a report of the search and of the selected design.

> **Hidden code.** Only the configuration cells (Section 2) and the cell that launches the single run (Section 8) show their code. The other code cells are collapsed and display a title; their code runs normally. To show the code of a cell, double-click it in Colab (or use *View → Show/hide code*), or click its collapsed input in Jupyter.

> **Note.** The surrogates are only reliable inside the parameter ranges used to build the training set (Table 4 of the paper). Candidate designs should be confirmed by a CFD simulation or by experiment before use.

### Notebook outline

1. Setup
2. Configuration ← *the only section a user normally needs to edit*
3. Physical model of the generator
4. Loading the surrogate ensembles
5. Ensemble prediction and uncertainty measures
6. Multi-criteria fitness function
7. Particle Swarm Optimization
8. Single run: results and convergence
9. Multiple independent runs and export
10. Search report

## 1. Setup

All dependencies (NumPy, SciPy, Matplotlib, pandas and PyTorch) come preinstalled in Google Colab.

In [ ]:
#@title Setup: imports and environment
import os
import io
import time
import shutil
import zipfile
import urllib.request
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy import stats

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

## 2. Configuration

### 2.1 Location of the trained surrogate models

Each ensemble is stored in a single PyTorch file that contains the weights of its 20 members together with their normalization statistics and some metadata (input order, input transforms, class names, output units):

```
<MODELS_SOURCE>/
└── models/
    ├── regime_ensemble.pt       # regime classifier ensemble
    └── diameter_ensemble.pt     # droplet-diameter regressor ensemble
```

The 20 members of each ensemble share the same architecture; how they were trained and selected is described in Section 4 of the paper.

By default the models are read from the GitHub repository of this notebook. When `MODELS_SOURCE` is a URL, the two files are downloaded once and cached in `MODELS_CACHE_DIR`.

**Private repository.** While the repository is private, downloading the models requires a GitHub personal access token with read access to it. The token is never written in the notebook: in Colab, add it as a secret named `GITHUB_TOKEN` (key icon in the left panel) and grant this notebook access to it; elsewhere, set the environment variable `GITHUB_TOKEN`. No token is needed once the repository is public, or when `MODELS_SOURCE` is a local folder.

In [ ]:
# Local folder (default: the folder that holds this notebook), a base URL,
# or a .zip file (local path or URL). Examples:
#   MODELS_SOURCE = "."
#   MODELS_SOURCE = "/content/drive/MyDrive/colab"
#   MODELS_SOURCE = "https://raw.githubusercontent.com/<user>/<repo>/main"
#   MODELS_SOURCE = "https://github.com/<user>/<repo>/archive/refs/heads/main.zip"
MODELS_SOURCE = "https://raw.githubusercontent.com/mfgerard/PSO_inverse_design/main"

# Name of the Colab secret (or environment variable) that holds a GitHub token.
# Needed only while the repository is private; never paste the token itself here.
GITHUB_TOKEN_NAME = "GITHUB_TOKEN"

MODELS_CACHE_DIR = "./models_cache"   # used only for remote sources

# Ensemble files, relative to MODELS_SOURCE
REGIME_ENSEMBLE_FILE = "models/regime_ensemble.pt"
DIAMETER_ENSEMBLE_FILE = "models/diameter_ensemble.pt"

### 2.2 Design problem

The surrogates take seven inputs: the outer-phase capillary number $Ca_{out}$, the inner-phase Weber number $We_{in}$, the velocities $u_{out}$ and $u_{in}$, the viscosities $\mu_{out}$ and $\mu_{in}$ and the inner capillary radius $r_{in}$. Since $Ca_{out}$ and $We_{in}$ are functions of the other quantities (Section 3), the PSO searches a **five-dimensional** space:

| `DESIGN_MODE` | searched variables |
|---|---|
| `"dimensionless"` (paper) | $Ca_{out},\ We_{in},\ \mu_{out},\ \mu_{in},\ r_{in}$ |
| `"velocities"` | $u_{out},\ u_{in},\ \mu_{out},\ \mu_{in},\ r_{in}$ |

The variables not searched are computed from the searched ones.

* `TARGET_DIAMETER_UM`: target droplet diameter in µm. The paper validates targets between 21.75 and 248.8 µm (Table 6).
* `BOUNDS`: search ranges. The defaults are the ranges of the training set (Table 4). They can be narrowed but should not be widened: outside them the surrogates extrapolate.
* `FIXED`: parameters held at a given value during the search, for example a fixed inner capillary radius or the viscosities of fluids already chosen. Use `None` to leave a parameter free. In `"dimensionless"` mode, `u_out` and `u_in` cannot be fixed (they are derived), and the same holds for `C_out` and `W_in` in `"velocities"` mode.

All values are in SI units except the target diameter.

In [ ]:
TARGET_DIAMETER_UM = 64.26          # target droplet diameter [µm]

DESIGN_MODE = "dimensionless"       # "dimensionless" (Ca_out, We_in) or "velocities" (u_out, u_in)

# Search ranges (training-set ranges, Table 4 of the paper)
BOUNDS = {
    "C_out":     (0.047,     0.8),       # outer capillary number Ca_out [-]
    "W_in":      (0.035,     20.0),      # inner Weber number We_in [-]
    "u_out":     (0.0199242, 3.59293),   # outer-phase velocity [m/s]
    "u_in":      (0.168162,  8.95161),   # inner-phase velocity [m/s]
    "mu_out":    (0.00890,   0.09500),   # outer-phase viscosity [Pa s]
    "mu_in":     (0.00089,   0.08900),   # inner-phase viscosity [Pa s]
    "radius_in": (0.5e-5,    2.5e-5),    # inner capillary radius [m]
}

# Parameters held fixed during the search (None = free)
FIXED = {
    "C_out":     None,
    "W_in":      None,
    "u_out":     None,
    "u_in":      None,
    "mu_out":    None,
    "mu_in":     None,
    "radius_in": None,    # e.g. 2.5e-5 to fix the inner radius at 25 µm
}

### 2.3 Fixed physical properties

These properties are fixed in the simulations used to train the surrogates (Section 3.2) and must not be changed unless the surrogates are retrained.

In [ ]:
RHO_IN     = 997.0     # inner (dispersed) phase density [kg/m^3]
RHO_OUT    = 965.0     # outer (continuous) phase density [kg/m^3]
RADIUS_OUT = 10e-5     # outer capillary radius [m] (100 µm)
SIGMA      = 40e-3     # interfacial tension [N/m]

### 2.4 PSO hyperparameters

Defaults from Section 5.1 of the paper.

| symbol | variable | meaning |
|---|---|---|
| $N$ | `N_PARTICLES` | swarm size |
| $T_{max}$ | `MAX_ITER` | maximum number of iterations |
| $P$ | `PATIENCE` | iterations without significant improvement before stopping |
| $\varepsilon$ | `TOL_IMPROVEMENT` | minimum relative decrease of the global-best fitness that resets the patience counter |
| $c_1,\ c_2$ | `C1`, `C2` | cognitive and social acceleration coefficients |
| $\alpha_b$ | `REBOUND` | factor applied to the velocity of a particle that leaves the domain |
| $\delta$ | `VELOCITY_CLAMP` | maximum velocity as a fraction of each variable's range (`None` disables clamping) |

In [ ]:
N_PARTICLES     = 50
MAX_ITER        = 500
PATIENCE        = 150
TOL_IMPROVEMENT = 0.10
C1              = 2.0
C2              = 2.0
REBOUND         = -0.7
# Velocity clamping |v_j| <= VELOCITY_CLAMP * (x_max_j - x_min_j).
# NOTE: Section 5.1 of the paper lists delta = 0.9, but the clamping step was disabled
# in the code that produced the results of the paper (only the initial velocities are
# clipped, see INIT_VEL_FRAC). None reproduces that code; set 0.9 to follow the text.
VELOCITY_CLAMP  = None
INIT_VEL_FRAC   = 0.5      # initial velocities are clipped to ±INIT_VEL_FRAC * range

SEED            = 42       # random seed of the first run (None = non-reproducible)
LOG_EVERY       = 20       # print progress every LOG_EVERY iterations (0 = silent)
N_RUNS          = 3        # independent runs in Section 9 (3 per target in the paper)
REPORT_RUN      = None     # run reported in Section 10 (1 ... N_RUNS); None = accepted run with the lowest fitness
MAX_REL_ERROR   = 0.05     # a run is accepted if |d_pred - target| / target <= MAX_REL_ERROR and the
                           # ensemble consensus is dripping; otherwise it is flagged as failed in Section 9
MAP_RESOLUTION  = 120      # grid points per axis of the regime/diameter map in Section 10

### 2.5 Fitness-function constants

Constants of the fitness terms in Section 5.2 (Eqs. 17–21). `WEIGHTS` multiplies the five terms $t_0,\dots,t_4$; the paper uses equal weights.

In [ ]:
ERR_SCALE      = 0.10    # relative diameter error mapped to tanh(1) ≈ 0.76 in t0 (Eq. 17)
P_THRESHOLD    = 0.75    # per-member probability threshold for "confident dripping" in t1 (Eq. 18)
CV_SCALE       = 0.13    # diameter CV mapped to tanh(1) ≈ 0.76 in t3 (Eq. 20)
BOUNDARY_EXP   = 3.0     # exponent of the boundary-proximity penalty t4 (Eq. 21)
WEIGHTS        = (1.0, 1.0, 1.0, 1.0, 1.0)   # weights of t0 ... t4
CONFIDENCE     = 0.95    # level of the t-Student confidence interval on the diameter

## 3. Physical model of the generator

In the coaxial generator the dispersed phase flows through the inner capillary (radius $r_{in}$) and the continuous phase flows through the outer capillary (radius $r_{out}$). The flow is described by the capillary number of the outer phase and the Weber number of the inner phase (Eqs. 1–2 of the paper), with the inner capillary diameter as the characteristic length of $We_{in}$:

$$
Ca_{out} = \frac{\mu_{out}\, u_{out}}{\sigma},
\qquad
We_{in} = \frac{\rho_{in}\, u_{in}^{2}\, (2 r_{in})}{\sigma}.
$$

Given one pair of variables ($Ca_{out}, We_{in}$ or $u_{out}, u_{in}$), the other follows:

$$
u_{out} = \frac{\sigma\, Ca_{out}}{\mu_{out}},
\qquad
u_{in} = \sqrt{\frac{\sigma\, We_{in}}{2\,\rho_{in}\, r_{in}}}.
$$

The function below builds the full seven-input vector that the surrogates expect from a five-dimensional PSO position.

In [ ]:
#@title Physical model: search variables and conversion to surrogate inputs
PARAM_NAMES = ["C_out", "W_in", "u_out", "u_in", "mu_out", "mu_in", "radius_in"]   # surrogate input order
PARAM_LABELS = {
    "C_out": r"$Ca_{out}$", "W_in": r"$We_{in}$",
    "u_out": r"$u_{out}$ [m/s]", "u_in": r"$u_{in}$ [m/s]",
    "mu_out": r"$\mu_{out}$ [Pa s]", "mu_in": r"$\mu_{in}$ [Pa s]",
    "radius_in": r"$r_{in}$ [m]",
}

if DESIGN_MODE == "dimensionless":
    SEARCH_VARS = ["C_out", "W_in", "mu_out", "mu_in", "radius_in"]
elif DESIGN_MODE == "velocities":
    SEARCH_VARS = ["u_out", "u_in", "mu_out", "mu_in", "radius_in"]
else:
    raise ValueError("DESIGN_MODE must be 'dimensionless' or 'velocities'")


def to_full_inputs(X: np.ndarray) -> np.ndarray:
    # Map PSO positions (n, 5) in the order SEARCH_VARS to surrogate inputs (n, 7) in the order PARAM_NAMES.
    X = np.atleast_2d(X)
    mu_out, mu_in, r_in = X[:, 2], X[:, 3], X[:, 4]
    if DESIGN_MODE == "dimensionless":
        C_out, W_in = X[:, 0], X[:, 1]
        u_out = SIGMA * C_out / mu_out
        u_in = np.sqrt(SIGMA * W_in / (RHO_IN * 2.0 * r_in))
    else:
        u_out, u_in = X[:, 0], X[:, 1]
        C_out = u_out * mu_out / SIGMA
        W_in = u_in**2 * RHO_IN * 2.0 * r_in / SIGMA
    return np.column_stack([C_out, W_in, u_out, u_in, mu_out, mu_in, r_in])


# Search domain D (5 x 2) with fixed parameters collapsed to a single value
for name, value in FIXED.items():
    if value is not None and name not in SEARCH_VARS:
        raise ValueError(f"'{name}' is derived in DESIGN_MODE='{DESIGN_MODE}' and cannot be fixed. "
                         f"Fixable parameters: {SEARCH_VARS}")
D = np.array([(FIXED[v], FIXED[v]) if FIXED[v] is not None else BOUNDS[v] for v in SEARCH_VARS], dtype=float)
TARGET_DIAMETER = TARGET_DIAMETER_UM * 1e-6   # [m]; the regressor predicts diameters in metres

pd.DataFrame(D, index=SEARCH_VARS, columns=["lower", "upper"]).assign(
    fixed=[FIXED[v] is not None for v in SEARCH_VARS])

## 4. Loading the surrogate ensembles

The two architectures were selected by hyperparameter optimization (Section 4.2, Tables 1–2):

* **Regime classifier** $\phi_r$: 7 → 30 (ReLU, dropout 0.2) → 3 logits, followed by a softmax over the three regimes.
* **Diameter regressor** $\phi_d$: 7 → 40 (ReLU, dropout 0.1) → 5 (ReLU) → 1.

Both networks receive the seven inputs with the two viscosities log-transformed and then standardized with each member's own statistics, $\tilde{x} = (x - \bar{x}_{train}) / s_{train}$. The regressor output is de-standardized to obtain the diameter in metres. The list of log-transformed inputs and all statistics are read from the ensemble files.

In [ ]:
#@title Surrogate architectures
class RegimeNet(nn.Module):
    # Regime classifier phi_r: 7-30-3
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 30), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(30, 3),
        )

    def forward(self, x):
        return self.net(x)


class DiameterNet(nn.Module):
    # Droplet-diameter regressor phi_d: 7-40-5-1
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(7, 40), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(40, 5), nn.ReLU(),
            nn.Linear(5, 1),
        )

    def forward(self, x):
        return self.model(x)

The helper `resolve()` returns a local path for each ensemble file. For a local folder it joins paths. For a base URL it downloads the file into the cache. For a `.zip` archive it downloads (if remote) and extracts the archive once, and then looks for the expected folder tree inside it, so a GitHub archive with an extra top-level folder also works.

In [ ]:
#@title Model source: local folder, URL or zip
def _is_url(s: str) -> bool:
    return s.startswith(("http://", "https://"))


def _github_token():
    # Token for private GitHub repositories: Colab secret first, then environment variable.
    try:
        from google.colab import userdata
        return userdata.get(GITHUB_TOKEN_NAME)
    except Exception:
        return os.environ.get(GITHUB_TOKEN_NAME)


def _download(url: str, dest: str) -> str:
    if not os.path.exists(dest):
        os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
        req = urllib.request.Request(url)
        token = _github_token() if "github" in url else None
        if token:
            req.add_header("Authorization", f"token {token}")
        tmp = dest + ".part"
        try:
            with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f)
        except urllib.error.HTTPError as e:
            hint = ""
            if e.code == 404 and "github" in url:
                hint = (" If the repository is private, provide a GitHub token in the Colab secret "
                        f"(or environment variable) '{GITHUB_TOKEN_NAME}'"
                        + (" (a token was found: check that it can read the repository)." if token else "."))
            raise RuntimeError(f"Could not download {url} (HTTP {e.code}).{hint}") from None
        os.replace(tmp, dest)
    return dest


def _prepare_source(source: str) -> str:
    # Return a local root folder, or a base URL, from which model files are resolved.
    if source.lower().endswith(".zip"):
        zip_path = _download(source, os.path.join(MODELS_CACHE_DIR, "models.zip")) if _is_url(source) else source
        extract_dir = os.path.join(MODELS_CACHE_DIR, "unzipped")
        if not os.path.isdir(extract_dir):
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(extract_dir)
        # Locate the folder that contains the ensemble files (handles extra top-level folders)
        for dirpath, _, _ in os.walk(extract_dir):
            if os.path.isfile(os.path.join(dirpath, REGIME_ENSEMBLE_FILE)):
                return dirpath
        raise FileNotFoundError(f"'{REGIME_ENSEMBLE_FILE}' not found inside {source}")
    return source.rstrip("/")


_SOURCE_ROOT = _prepare_source(MODELS_SOURCE)


def resolve(rel_path: str) -> str:
    # Local path of a model file given its path relative to MODELS_SOURCE.
    if _is_url(_SOURCE_ROOT):
        return _download(f"{_SOURCE_ROOT}/{rel_path}", os.path.join(MODELS_CACHE_DIR, rel_path))
    path = os.path.join(_SOURCE_ROOT, rel_path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model file not found: {path}. Check MODELS_SOURCE.")
    return path

In [ ]:
#@title Load the two ensembles
@dataclass
class Member:
    model: nn.Module
    X_mean: np.ndarray
    X_std: np.ndarray
    y_mean: float = 0.0
    y_std: float = 1.0
    name: str = ""


def load_ensemble(rel_path: str, model_class) -> tuple:
    # Load a packed ensemble file; returns (list of members, metadata dict)
    pack = torch.load(resolve(rel_path), map_location=device, weights_only=True)
    expected = model_class.__name__
    if pack["architecture"]["name"] != expected:
        raise ValueError(f"{rel_path} holds a {pack['architecture']['name']} ensemble, expected {expected}")
    if pack["input_names"] != PARAM_NAMES or pack["log_inputs"] != ["mu_out", "mu_in"]:
        raise ValueError(f"{rel_path}: unexpected input definition {pack['input_names']} / {pack['log_inputs']}")

    regression = "y_mean" in pack
    members = []
    for i, k in enumerate(pack["member_ids"]):
        model = model_class().to(device)
        model.load_state_dict(pack["state_dicts"][i])
        model.eval()
        m = Member(model=model,
                   X_mean=pack["X_mean"][i].cpu().numpy().astype(np.float32),
                   X_std=pack["X_std"][i].cpu().numpy().astype(np.float32),
                   name=str(k))
        if regression:
            m.y_mean = float(pack["y_mean"][i])
            m.y_std = float(pack["y_std"][i])
        members.append(m)
    meta = {k: v for k, v in pack.items() if k not in ("state_dicts", "X_mean", "X_std", "y_mean", "y_std")}
    return members, meta


t0_ = time.time()
ENSEMBLE_REG, META_REG = load_ensemble(REGIME_ENSEMBLE_FILE, RegimeNet)
ENSEMBLE_DIA, META_DIA = load_ensemble(DIAMETER_ENSEMBLE_FILE, DiameterNet)

REGIME_NAMES = dict(enumerate(META_REG["output"]["class_names"]))
DRIPPING_CLASS = META_REG["output"]["dripping_class"]

print(f"Regime ensemble:   {len(ENSEMBLE_REG)} members | classes: {list(REGIME_NAMES.values())}")
print(f"Diameter ensemble: {len(ENSEMBLE_DIA)} members | output: {META_DIA['output']['name']} [{META_DIA['output']['units']}]")
print(f"Loaded in {time.time() - t0_:.1f} s from {MODELS_SOURCE!r}")

## 5. Ensemble prediction and uncertainty measures

For a batch of candidate designs, every member of both ensembles is evaluated. From the 20 regime predictions we obtain:

* the **consensus class** $\hat{c}$, by majority vote, with ties broken by the class probabilities summed over members;
* the number of votes for dripping, $N^{DR}_{votes}$, and the number of members that predict dripping with probability above $p_{th}$;
* the ensemble **consensus** $c(\mathbf{x}) = \frac{1}{N}\max_r \sum_{i=1}^{N} \mathbb{I}[\hat{y}_i(\mathbf{x}) = r]$ (Eq. 11).

From the 20 diameter predictions $d_k$ we obtain the ensemble mean $\bar{d}$, the standard deviation $s_d$, the **coefficient of variation** $c_v = s_d/|\bar{d}|$ (the absolute value guards against negative means that the regressor can output far outside the dripping region), and a two-sided t-Student confidence interval for the mean, $\bar{d} \pm t_{1-\alpha/2,\,N-1}\, s_d / \sqrt{N}$.

In [ ]:
#@title Ensemble prediction and uncertainty measures
def _standardize(cases: np.ndarray, m: Member) -> torch.Tensor:
    x = cases.astype(np.float32, copy=True)
    x[:, 4] = np.log(x[:, 4] + 1e-12)   # log(mu_out)
    x[:, 5] = np.log(x[:, 5] + 1e-12)   # log(mu_in)
    return torch.from_numpy((x - m.X_mean) / m.X_std).to(device)


@torch.no_grad()
def member_predictions(cases: np.ndarray):
    # Per-member outputs for full 7-input cases (n, 7):
    # regime probabilities (M_r, n, 3) and diameters in metres (M_d, n).
    cases = np.atleast_2d(cases)
    probs = np.stack([torch.softmax(m.model(_standardize(cases, m)), dim=1).cpu().numpy()
                      for m in ENSEMBLE_REG])
    diams = np.stack([m.model(_standardize(cases, m)).squeeze(-1).cpu().numpy() * m.y_std + m.y_mean
                      for m in ENSEMBLE_DIA])
    return probs, diams


def ensemble_predict(cases: np.ndarray) -> dict:
    # Ensemble-level predictions and uncertainty measures for a batch of cases (n, 7).
    probs, diams = member_predictions(cases)

    n_reg, n_cases, n_cls = probs.shape
    classes = probs.argmax(axis=2)                                               # (M_r, n)
    votes = np.stack([(classes == c).sum(axis=0) for c in range(n_cls)], axis=1) # (n, 3)
    max_votes = votes.max(axis=1)
    # Majority vote; ties broken by the accumulated probabilities of the tied classes
    tie_score = np.where(votes == max_votes[:, None], probs.sum(axis=0), -np.inf)
    consensus_class = tie_score.argmax(axis=1)

    d_mean = diams.mean(axis=0)
    d_std = diams.std(axis=0)
    n_dia = diams.shape[0]
    t_val = stats.t.ppf(0.5 + CONFIDENCE / 2, df=n_dia - 1)

    return {
        "class": consensus_class,
        "consensus": max_votes / n_reg,                                   # Eq. 11
        "votes_dr": votes[:, DRIPPING_CLASS],
        "confident_dr": (probs[:, :, DRIPPING_CLASS] > P_THRESHOLD).sum(axis=0),
        "p_dr_mean": probs[:, :, DRIPPING_CLASS].mean(axis=0),
        "p_dr_std": probs[:, :, DRIPPING_CLASS].std(axis=0),
        "n_reg": n_reg,
        "d_mean": d_mean,
        "d_std": d_std,
        "d_cv": d_std / (np.abs(d_mean) + 1e-20),   # |mean|: a negative (extrapolated) mean must not give a negative CV
        "d_ci": t_val * d_std / np.sqrt(n_dia),
    }

Quick check: prediction for case 1 of Table 6 in the paper ($Ca_{out}=0.323$, $We_{in}=0.164$, $\mu_{out}=22.6$ mPa s, $\mu_{in}=0.890$ mPa s, $r_{in}=5.01$ µm), whose simulated droplet diameter is 21.44 µm. The values in Table 6 are rounded, so the prediction will be close to, but not exactly, the reported 21.75 µm.

In [ ]:
#@title Example: prediction for case 1 of Table 6
def _example_case():
    C_out, W_in, mu_out, mu_in, r_in = 0.323, 0.164, 22.6e-3, 0.890e-3, 5.01e-6
    u_out = SIGMA * C_out / mu_out
    u_in = np.sqrt(SIGMA * W_in / (RHO_IN * 2 * r_in))
    return np.array([[C_out, W_in, u_out, u_in, mu_out, mu_in, r_in]])

_p = ensemble_predict(_example_case())
print(f"Regime: {REGIME_NAMES[int(_p['class'][0])]} (consensus {_p['consensus'][0]:.2f})")
print(f"Diameter: {_p['d_mean'][0]*1e6:.2f} ± {_p['d_ci'][0]*1e6:.2f} µm  (CV = {_p['d_cv'][0]*100:.2f} %)")

## 6. Multi-criteria fitness function

PSO minimizes (Eq. 16)

$$
F(\mathbf{x}) = t_0 + t_1 + t_2 + t_3 + t_4 ,
$$

where each term lies in $[0, 1]$:

* **Diameter error** (Eq. 17): $\displaystyle t_0 = \tanh\!\left(\frac{|d_{target} - \bar{d}|}{0.1\, d_{target}}\right)$. A relative error of 1 % gives 0.10, 10 % gives 0.76 and 20 % gives 0.96.
* **Regime confidence** (Eq. 18): $\displaystyle t_1 = 1 - \frac{1}{|\mathcal{E}_r|}\sum_{k} \mathbb{I}\!\left[p^{DR}_k > p_{th}\right]$, the fraction of classifier members that do **not** predict dripping with probability above $p_{th} = 0.75$.
* **Regime consensus** (Eq. 19): $\displaystyle t_2 = \left(1 - \frac{N^{DR}_{votes}}{|\mathcal{E}_r|}\right)\, \mathbb{I}\!\left[\hat{c} \neq DR\right]$. It is non-zero only when the consensus regime is a jetting regime, and then $t_2 \geq 0.5$.
* **Diameter uncertainty** (Eq. 20): $\displaystyle t_3 = \tanh\!\left(\frac{c_v}{0.13}\right)$, which penalizes disagreement among the diameter models.
* **Boundary proximity** (Eq. 21): $\displaystyle t_4 = \frac{1}{2}\left(|2\hat{x}_{1} - 1|^{3} + |2\hat{x}_{2} - 1|^{3}\right)$, where $\hat{x}_{1}, \hat{x}_{2} \in [0,1]$ are the first two search variables ($Ca_{out}$ and $We_{in}$, or $u_{out}$ and $u_{in}$) normalized to their ranges. It keeps the search away from the edges of the training domain, where the surrogates are less reliable. A fixed variable contributes zero.

The diameter regressor was trained only on dripping cases, so $t_1$ and $t_2$ steer the swarm towards designs where its prediction is meaningful.

In [ ]:
#@title Fitness function
TERM_NAMES = ["t0 diameter error", "t1 regime confidence", "t2 regime consensus",
              "t3 diameter uncertainty", "t4 boundary proximity"]


def fitness(X: np.ndarray):
    # Fitness of PSO positions X (n, 5). Returns F (n,), terms (n, 5) and the ensemble predictions.
    pred = ensemble_predict(to_full_inputs(X))

    rel_err = np.abs(TARGET_DIAMETER - pred["d_mean"]) / TARGET_DIAMETER
    t0 = np.tanh(rel_err / ERR_SCALE)
    t1 = 1.0 - pred["confident_dr"] / pred["n_reg"]
    t2 = (1.0 - pred["votes_dr"] / pred["n_reg"]) * (pred["class"] != DRIPPING_CLASS)
    t3 = np.tanh(pred["d_cv"] / CV_SCALE)

    penalties = []
    for j in (0, 1):
        span = D[j, 1] - D[j, 0]
        if span > 0:
            p = (X[:, j] - D[j, 0]) / span
            penalties.append(np.abs(2.0 * p - 1.0) ** BOUNDARY_EXP)
        else:   # fixed variable: no boundary penalty
            penalties.append(np.zeros(len(X)))
    t4 = 0.5 * (penalties[0] + penalties[1])

    terms = np.column_stack([t0, t1, t2, t3, t4]) * np.asarray(WEIGHTS)
    return terms.sum(axis=1), terms, pred

## 7. Particle Swarm Optimization

A global-best PSO (Section 3.3) updates each particle $i$ and dimension $j$ as (Eqs. 8–9)

$$
v_{ij}(t+1) = w_{ij}(t)\, v_{ij}(t) + c_1 r_{1j}(t)\,[y_{ij}(t) - x_{ij}(t)] + c_2 r_{2j}(t)\,[\hat{y}_{j}(t) - x_{ij}(t)],
\qquad
\mathbf{x}_i(t+1) = \mathbf{x}_i(t) + \mathbf{v}_i(t+1),
$$

where $\mathbf{y}_i$ is the personal best of particle $i$, $\hat{\mathbf{y}}$ is the global best and $r_{1j}, r_{2j} \sim U(0,1)$. The implementation includes the modifications described in Section 5.1:

* **random inertia**: $w_{ij}(t) \sim U(0,1)$ is drawn for every particle, dimension and iteration;
* **initialization**: uniform positions in $D$ and velocities uniform in $[-\Delta x_j, \Delta x_j]$, clipped to $\pm 0.5\,\Delta x_j$, with $\Delta x_j = x_{j}^{max} - x_{j}^{min}$;
* **boundary handling**: a particle that leaves $D$ is placed on the boundary, and its velocity component is multiplied by the rebound factor $\alpha_b = -0.7$;
* optional **velocity clamping** $|v_{ij}| \le \delta\, \Delta x_j$ (`VELOCITY_CLAMP`);
* **stopping**: after $T_{max}$ iterations, or after $P$ iterations without an improvement of the global best fitness larger than $\varepsilon = 10\,\%$ in relative terms.

In [ ]:
#@title PSO algorithm
@dataclass
class PSOResult:
    seed: object                # random seed of the run
    x_best: np.ndarray          # global best position (5,)
    f_best: float
    terms_best: np.ndarray      # (5,)
    pred_best: dict             # ensemble predictions at x_best
    iterations: int
    stop_reason: str
    history: dict = field(default_factory=dict)


def _pick(pred: dict, i: int) -> dict:
    return {k: (v[i] if isinstance(v, np.ndarray) else v) for k, v in pred.items()}


def _log(it, x, f, terms, pred):
    lbl = SEARCH_VARS
    xs = " ".join(f"{lbl[j]}={x[j]:.4g}" for j in range(len(x)))
    print(f"it {it:4d} | F={f:.5f} | d={pred['d_mean']*1e6:8.3f} µm (CV {pred['d_cv']*100:5.2f} %) "
          f"| {REGIME_NAMES[int(pred['class'])].split()[0]:>9s} | {xs}")


def run_pso(seed=None, verbose=True) -> PSOResult:
    rng = np.random.default_rng(seed)
    dim = D.shape[0]
    lo, hi = D[:, 0], D[:, 1]
    span = hi - lo

    # Initialization
    X = lo + rng.random((N_PARTICLES, dim)) * span
    V = np.clip((2 * rng.random((N_PARTICLES, dim)) - 1) * span, -INIT_VEL_FRAC * span, INIT_VEL_FRAC * span)
    F, terms, pred = fitness(X)
    X_loc, F_loc = X.copy(), F.copy()
    g = int(np.argmin(F))
    x_glob, f_glob, t_glob, p_glob = X[g].copy(), F[g], terms[g].copy(), _pick(pred, g)

    hist = {"f_best": [f_glob], "terms_best": [t_glob], "x_best": [x_glob.copy()],
            "f_mean": [F.mean()], "swarm_init": X.copy(), "swarm": [X.copy()]}
    patience = 0
    stop_reason = f"maximum number of iterations ({MAX_ITER})"
    if verbose and LOG_EVERY:
        _log(0, x_glob, f_glob, t_glob, p_glob)

    it = 0
    for it in range(1, MAX_ITER + 1):
        W = rng.random((N_PARTICLES, dim))                # random inertia
        r1 = rng.random((N_PARTICLES, dim))
        r2 = rng.random((N_PARTICLES, dim))
        V = W * V + C1 * r1 * (X_loc - X) + C2 * r2 * (x_glob - X)
        if VELOCITY_CLAMP is not None:
            V = np.clip(V, -VELOCITY_CLAMP * span, VELOCITY_CLAMP * span)

        X_new = X + V
        out = (X_new < lo) | (X_new > hi)
        X = np.clip(X_new, lo, hi)
        V[out] *= REBOUND

        F, terms, pred = fitness(X)

        # Personal bests (Eq. 10)
        better = F < F_loc
        X_loc[better], F_loc[better] = X[better], F[better]

        # Global best and patience
        g = int(np.argmin(F))
        if F[g] < f_glob:
            if F[g] < f_glob * (1 - TOL_IMPROVEMENT):
                patience = 0
            x_glob, f_glob, t_glob, p_glob = X[g].copy(), F[g], terms[g].copy(), _pick(pred, g)
        else:
            patience += 1

        hist["f_best"].append(f_glob); hist["terms_best"].append(t_glob)
        hist["x_best"].append(x_glob.copy()); hist["f_mean"].append(F.mean())
        hist["swarm"].append(X.copy())

        if verbose and LOG_EVERY and it % LOG_EVERY == 0:
            _log(it, x_glob, f_glob, t_glob, p_glob)
        if patience >= PATIENCE:
            stop_reason = f"no significant improvement in {PATIENCE} iterations"
            break

    hist = {k: np.array(v) for k, v in hist.items()}
    hist["swarm_final"] = X.copy()
    return PSOResult(seed, x_glob, float(f_glob), t_glob, p_glob, it, stop_reason, hist)

## 8. Single run: results and convergence

In [ ]:
t_start = time.time()
result = run_pso(seed=SEED)
print(f"\nStopped after {result.iterations} iterations ({result.stop_reason}) in {time.time() - t_start:.1f} s")

In [ ]:
#@title Design table of the single run
def design_table(res: PSOResult) -> pd.DataFrame:
    full = to_full_inputs(res.x_best)[0]
    p = res.pred_best
    rows = {name: full[i] for i, name in enumerate(PARAM_NAMES)}
    rows.update({
        "target diameter [µm]": TARGET_DIAMETER_UM,
        "predicted diameter [µm]": p["d_mean"] * 1e6,
        f"{int(CONFIDENCE*100)}% CI half-width [µm]": p["d_ci"] * 1e6,
        "relative error [%]": 100 * (p["d_mean"] - TARGET_DIAMETER) / TARGET_DIAMETER,
        "diameter CV [%]": p["d_cv"] * 100,
        "regime": REGIME_NAMES[int(p["class"])],
        "regime consensus": p["consensus"],
        "mean P(dripping)": p["p_dr_mean"],
        "fitness F": res.f_best,
    })
    rows.update({n: v for n, v in zip(TERM_NAMES, res.terms_best)})
    return pd.DataFrame.from_dict(rows, orient="index", columns=["value"])

design_table(result)

**Convergence.** Left: fitness of the global best and mean fitness of the swarm. Right: contribution of each fitness term at the global best.

In [ ]:
#@title Plot: convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
h = result.history
axes[0].semilogy(h["f_best"], label="global best")
axes[0].semilogy(h["f_mean"], alpha=0.5, label="swarm mean")
axes[0].set(xlabel="iteration", ylabel="fitness F", title="Fitness")
axes[0].legend(); axes[0].grid(True, which="both", alpha=0.3)

for k, name in enumerate(TERM_NAMES):
    axes[1].plot(h["terms_best"][:, k], label=name)
axes[1].set(xlabel="iteration", ylabel="term value", title="Fitness terms of the global best")
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**Search trajectory.** Initial swarm (red), final swarm (blue) and path of the global best in the plane of the first two search variables, on log–log axes.

In [ ]:
#@title Plot: search trajectory
def plot_trajectory(res: PSOResult, ax=None):
    ax = ax or plt.subplots(figsize=(6, 5))[1]
    h = res.history
    ax.scatter(h["swarm_init"][:, 0], h["swarm_init"][:, 1], s=10, c="tab:red", alpha=0.3, label="initial swarm")
    ax.scatter(h["swarm_final"][:, 0], h["swarm_final"][:, 1], s=15, c="tab:blue", alpha=0.3, label="final swarm")
    ax.plot(h["x_best"][:, 0], h["x_best"][:, 1], "-o", ms=3, c="k", lw=1, label="global best path")
    ax.scatter(*res.x_best[:2], s=120, facecolors="gold", edgecolors="k", zorder=5, label="final design")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(BOUNDS[SEARCH_VARS[0]]); ax.set_ylim(BOUNDS[SEARCH_VARS[1]])
    ax.set_xlabel(PARAM_LABELS[SEARCH_VARS[0]]); ax.set_ylabel(PARAM_LABELS[SEARCH_VARS[1]])
    ax.set_title(f"PSO search, target {TARGET_DIAMETER_UM:g} µm")
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)
    return ax

plot_trajectory(result); plt.show()

## 9. Multiple independent runs and export

The fitness landscape is multimodal: different parameter combinations can produce the same droplet. Independent runs with different seeds therefore usually give different designs with similar fitness, which gives the designer several options to choose from (in the paper, three runs per target were simulated for validation). The table lists each run's design and its predicted performance, sorted by fitness, and is saved to CSV.

Occasionally a swarm converges prematurely to a poor design (in tests with the targets of Table 6, about 1 % of the runs). Each run is therefore checked against two acceptance criteria: the predicted diameter must be within `MAX_REL_ERROR` of the target, and the ensemble consensus must be the dripping regime. Runs that fail either criterion are marked in the `status` column and should not be used as designs; increase `N_RUNS` if few runs are accepted.

In [ ]:
#@title Run the independent searches, check them and export to CSV
def run_status(res: PSOResult) -> str:
    # Acceptance check of a run: diameter error and dripping consensus at the global best
    p = res.pred_best
    err = abs(p["d_mean"] - TARGET_DIAMETER) / TARGET_DIAMETER
    problems = []
    if err > MAX_REL_ERROR:
        problems.append(f"error {err:.2%} > {MAX_REL_ERROR:.2%}")
    if int(p["class"]) != DRIPPING_CLASS:
        problems.append(f"regime {REGIME_NAMES[int(p['class'])]}")
    return "accepted" if not problems else "FAILED: " + "; ".join(problems)


runs, run_results = [], []
for r in range(N_RUNS):
    seed = None if SEED is None else SEED + r
    res = run_pso(seed=seed, verbose=False)
    run_results.append(res)
    row = design_table(res)["value"].to_dict()
    status = run_status(res)
    row.update({"run": r + 1, "seed": seed, "iterations": res.iterations, "status": status})
    runs.append(row)
    print(f"run {r+1}/{N_RUNS}: F={res.f_best:.5f}, d={res.pred_best['d_mean']*1e6:.3f} µm, "
          f"{res.iterations} iterations -> {status}")

accepted = [i for i, res in enumerate(run_results) if run_status(res) == "accepted"]
if len(accepted) < N_RUNS:
    print(f"\nWARNING: {N_RUNS - len(accepted)} of {N_RUNS} runs failed the acceptance criteria "
          f"(error <= {MAX_REL_ERROR:.0%} and dripping consensus); do not use them as designs.")
if not accepted:
    print("WARNING: no run was accepted. Increase N_RUNS, or check BOUNDS, FIXED and the target.")

cols = ["status"] + [c for c in pd.DataFrame(runs).columns if c not in ("status", "run")]
runs_df = (pd.DataFrame(runs).set_index("run")[cols]
           .sort_values(["status", "fitness F"], key=lambda c: c != "accepted" if c.name == "status" else c))
csv_name = f"pso_designs_target_{TARGET_DIAMETER_UM:g}um.csv"
runs_df.to_csv(csv_name)
print(f"\nSaved to {csv_name}")
runs_df

In Colab, the CSV file can be downloaded from the file browser (left panel) or with:

```python
from google.colab import files
files.download(csv_name)
```

## 10. Search report

This section summarizes the search and evaluates the selected design in detail. By default it reports the accepted run of Section 9 with the lowest fitness (see the acceptance criteria in Section 9); `REPORT_RUN` (Section 2.4) selects another run. The report contains:

1. a **summary** of the design problem and of the search;
2. the **design table** with the seven surrogate inputs of the solution, the ranges in `BOUNDS` (by default, the training ranges of Table 4) and the position of each value within its range (on a linear scale, as in the boundary term $t_4$). The formatting of each row shows how the value was obtained:
   * **orange background, bold**: value **fixed in advance** by the user in `FIXED` (not searched by the PSO);
   * no highlight: value **found by the PSO**;
   * *grey italics*: value **derived** from the others through the definitions of $Ca_{out}$ and $We_{in}$ (Section 3).

   A position below 0 % or above 100 % (in red) means that the value lies outside its range; when the ranges are the training ranges, the surrogate prediction for that design is an extrapolation;
3. the **surrogate assessment** of the solution: the design is passed through every member of both ensembles to obtain the predicted droplet diameter with its confidence interval, and the confidence that the generator operates in the dripping regime;
4. **figures**: the exploration of the swarm in the $Ca_{out}$–$We_{in}$ plane, a regime and diameter map around the solution, the per-member predictions at the solution, and a comparison of all runs.

In [ ]:
#@title Report: summary
from IPython.display import display, Markdown
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

if REPORT_RUN is not None:
    rep_idx = REPORT_RUN - 1
else:
    pool = accepted or list(range(N_RUNS))        # fall back to all runs if none was accepted
    rep_idx = min(pool, key=lambda i: run_results[i].f_best)
rep_res = run_results[rep_idx]
rep_status = run_status(rep_res)
if rep_status != "accepted":
    print(f"WARNING: the reported run {rep_idx + 1} did not pass the acceptance criteria ({rep_status}).")
rep_full = to_full_inputs(rep_res.x_best)[0]          # the 7 surrogate inputs of the solution
LEVEL = f"{CONFIDENCE:.0%}"

free_vars = [v for v in SEARCH_VARS if FIXED[v] is None]
fixed_vars = {v: FIXED[v] for v in SEARCH_VARS if FIXED[v] is not None}
n_evals = sum(len(r.history["swarm"]) * N_PARTICLES for r in run_results)

display(Markdown(f'''
### 10.1 Summary

| | |
|---|---|
| **Target droplet diameter** | {TARGET_DIAMETER_UM:g} µm |
| **Design mode** | `{DESIGN_MODE}`: search over {", ".join(SEARCH_VARS)} |
| **Optimized parameters** | {", ".join(free_vars) or "none"} |
| **Fixed parameters** | {", ".join(f"{k} = {v:g}" for k, v in fixed_vars.items()) or "none"} |
| **Search effort** | {N_RUNS} independent runs × {N_PARTICLES} particles, {n_evals:,} fitness evaluations |
| **Accepted runs** | {len(accepted)} of {N_RUNS} (error ≤ {MAX_REL_ERROR:.0%} and dripping consensus) |
| **Reported run** | run {rep_idx + 1} of {N_RUNS} (seed {rep_res.seed}, {rep_status}): {rep_res.iterations} iterations, stopped by {rep_res.stop_reason} |
| **Fitness of the solution** | F = {rep_res.f_best:.4f} |
'''))

### 10.2 Design table

In [ ]:
#@title Report: design table
PARAM_INFO = {
    "C_out":     ("outer capillary number", "–"),
    "W_in":      ("inner Weber number", "–"),
    "u_out":     ("outer-phase velocity", "m/s"),
    "u_in":      ("inner-phase velocity", "m/s"),
    "mu_out":    ("outer-phase viscosity", "Pa s"),
    "mu_in":     ("inner-phase viscosity", "Pa s"),
    "radius_in": ("inner capillary radius", "m"),
}


def parameter_table(full: np.ndarray) -> pd.DataFrame:
    rows = []
    for i, name in enumerate(PARAM_NAMES):
        lo, hi = BOUNDS[name]
        if name in SEARCH_VARS:
            origin = "fixed" if FIXED[name] is not None else "optimized"
        else:
            origin = "derived"
        rows.append({"parameter": name, "description": PARAM_INFO[name][0], "value": full[i],
                     "units": PARAM_INFO[name][1], "range min": lo, "range max": hi,
                     "position in range, linear [%]": 100 * (full[i] - lo) / (hi - lo), "origin": origin})
    return pd.DataFrame(rows).set_index("parameter")


def style_parameter_table(df: pd.DataFrame):
    def by_origin(row):
        if row["origin"] == "fixed":
            return ["background-color: #ffe0b2; color: black; font-weight: bold"] * len(row)
        if row["origin"] == "derived":
            return ["color: #808080; font-style: italic"] * len(row)
        return [""] * len(row)

    def out_of_range(v):
        return "color: #c62828; font-weight: bold" if (v < 0 or v > 100) else ""

    st = df.style.apply(by_origin, axis=1)
    st = (st.map if hasattr(st, "map") else st.applymap)(out_of_range, subset=["position in range, linear [%]"])
    return st.format({"value": "{:.4g}", "range min": "{:.4g}", "range max": "{:.4g}",
                      "position in range, linear [%]": "{:.1f}"})


design_df = parameter_table(rep_full)
display(style_parameter_table(design_df))

### 10.3 Surrogate assessment of the solution

The solution is evaluated by each member of both ensembles.

* **Droplet diameter.** The prediction is the mean $\bar{d}$ of the 20 diameter members. Its confidence interval, $\bar{d} \pm t_{1-\alpha/2,\,N-1}\, s_d/\sqrt{N}$, measures how well the members agree (the epistemic uncertainty of the surrogate). It does not include the error of the surrogate with respect to CFD or experiments; in the paper, the median absolute error against CFD for the PSO designs was 2.77 % (Section 5.3).
* **Dripping regime.** The confidence is given by the number of classifier members that vote for dripping (the consensus $c$ of Eq. 11), the number of members that predict dripping with probability above $p_{th}$, and the mean probability of dripping over the members with its confidence interval. The paper treats a dripping prediction as reliable when $c \geq 0.9$, that is, at least 18 of 20 votes (Section 4.3.2).

In [ ]:
#@title Report: surrogate assessment
probs_m, diams_m = member_predictions(rep_full[None, :])
probs_m = probs_m[:, 0, :]                 # (M_r, 3) class probabilities per member
diams_m = diams_m[:, 0] * 1e6              # (M_d,) diameters per member [µm]
n_r, n_d = len(probs_m), len(diams_m)

d_mean, d_sd = diams_m.mean(), diams_m.std()
d_ci = stats.t.ppf(0.5 + CONFIDENCE / 2, df=n_d - 1) * d_sd / np.sqrt(n_d)
p_dr = probs_m[:, DRIPPING_CLASS]
p_ci = stats.t.ppf(0.5 + CONFIDENCE / 2, df=n_r - 1) * p_dr.std() / np.sqrt(n_r)
member_cls = probs_m.argmax(axis=1)
votes_dr = int((member_cls == DRIPPING_CLASS).sum())
confident_dr = int((p_dr > P_THRESHOLD).sum())
cons_cls = int(ensemble_predict(rep_full[None, :])["class"][0])
rel_err = 100 * (d_mean - TARGET_DIAMETER_UM) / TARGET_DIAMETER_UM

assessment = pd.DataFrame.from_dict({
    "target diameter [µm]": f"{TARGET_DIAMETER_UM:.3f}",
    "predicted diameter (ensemble mean) [µm]": f"{d_mean:.3f}",
    f"{LEVEL} confidence interval [µm]": f"[{d_mean - d_ci:.3f}, {d_mean + d_ci:.3f}]  (±{d_ci:.3f} µm, ±{100 * d_ci / d_mean:.2f} %)",
    "target inside the confidence interval": "yes" if abs(TARGET_DIAMETER_UM - d_mean) <= d_ci else "no",
    "relative error of the prediction [%]": f"{rel_err:+.3f}",
    "spread of the members: std / CV": f"{d_sd:.3f} µm / {100 * d_sd / abs(d_mean):.2f} %",
    "range of the members [µm]": f"[{diams_m.min():.3f}, {diams_m.max():.3f}]",
    "consensus regime": REGIME_NAMES[cons_cls],
    "members voting dripping": f"{votes_dr} / {n_r}  (consensus c = {votes_dr / n_r:.2f})",
    f"members with P(dripping) > {P_THRESHOLD:g}": f"{confident_dr} / {n_r}",
    "P(dripping), mean over members": f"{p_dr.mean():.4f}  ({LEVEL} CI [{max(0, p_dr.mean() - p_ci):.4f}, {min(1, p_dr.mean() + p_ci):.4f}])",
    "P(dripping), lowest member": f"{p_dr.min():.4f}",
}, orient="index", columns=["value"])
display(assessment)

reliable_dr = cons_cls == DRIPPING_CLASS and votes_dr / n_r >= 0.9 and p_dr.mean() > P_THRESHOLD
if reliable_dr:
    regime_msg = (f"✅ **Dripping regime predicted with high confidence:** {votes_dr} of {n_r} members vote for dripping "
                  f"(c ≥ 0.9) and the mean probability of dripping is {p_dr.mean():.1%}.")
else:
    regime_msg = (f"⚠️ **Dripping regime not reliably predicted:** {votes_dr} of {n_r} members vote for dripping "
                  f"and the mean probability of dripping is {p_dr.mean():.1%}. The diameter prediction is only "
                  f"meaningful in the dripping regime; consider another run or other bounds.")
diam_msg = (f"**Predicted diameter:** {d_mean:.2f} µm, {LEVEL} CI [{d_mean - d_ci:.2f}, {d_mean + d_ci:.2f}] µm "
            f"for a target of {TARGET_DIAMETER_UM:g} µm (relative error {rel_err:+.2f} %, member CV {100 * d_sd / abs(d_mean):.1f} %).")
display(Markdown(regime_msg + "\n\n" + diam_msg))

### 10.4 Exploration of the design space

**Left: exploration in the $Ca_{out}$–$We_{in}$ plane.** Every position evaluated during the reported run, coloured by iteration, together with the trajectory of the global best (black line; white circles mark the iterations where it changed), its initial position (red square) and the solution (star). The dashed rectangle is the search range. In `"velocities"` mode the positions are converted to $Ca_{out}$ and $We_{in}$ before plotting.

**Right: regime and diameter map around the solution.** A two-dimensional slice of the design space in which $\mu_{out}$, $\mu_{in}$ and $r_{in}$ are held at the values of the solution. The background shows the consensus regime predicted by the classifier ensemble; the thin lines are iso-lines of the predicted diameter (drawn only in the dripping region); the thick black line is the iso-line of the target diameter, so every point on it is an alternative design with the same predicted diameter for these fluids and this capillary; the dotted line is where the mean probability of dripping equals $p_{th}$. The fitness favours points of the target iso-line that are far from the regime boundaries, where the ensemble agrees, and far from the edges of the domain.

In [ ]:
#@title Report: exploration and regime/diameter map
CLASS_LIGHT = ["#cfe8ff", "#ffe0b2", "#ffcdd2"]   # map background per regime
CLASS_DARK = ["#1f77b4", "#ff9f40", "#d62728"]    # bars and markers per regime


def ca_we(X: np.ndarray):
    full = to_full_inputs(X)
    return full[:, 0], full[:, 1]


def log_limits(values, bounds, pad=1.15):
    lo = min(np.min(values), bounds[0]) / pad
    hi = max(np.max(values), bounds[1]) * pad
    return lo, hi


h = rep_res.history
swarm = h["swarm"]                                            # (iterations + 1, N, 5)
ca_all, we_all = ca_we(swarm.reshape(-1, swarm.shape[-1]))
it_all = np.repeat(np.arange(swarm.shape[0]), swarm.shape[1])
ca_best, we_best = ca_we(h["x_best"])
changed = np.r_[True, np.any(np.diff(h["x_best"], axis=0) != 0, axis=1)]
ca_sol, we_sol = rep_full[0], rep_full[1]
(ca_lo, ca_hi), (we_lo, we_hi) = BOUNDS["C_out"], BOUNDS["W_in"]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# (a) exploration
ax = axes[0]
sc = ax.scatter(ca_all, we_all, c=it_all, cmap="viridis", s=6, alpha=0.35, linewidths=0, rasterized=True)
fig.colorbar(sc, ax=ax, label="iteration")
ax.add_patch(Rectangle((ca_lo, we_lo), ca_hi - ca_lo, we_hi - we_lo, fill=False, ls="--", ec="grey", lw=1))
ax.plot(ca_best, we_best, "-", c="k", lw=1.2, zorder=3)
ax.scatter(ca_best[changed], we_best[changed], s=20, c="white", edgecolors="k", zorder=4)
ax.scatter(ca_best[0], we_best[0], marker="s", s=70, c="tab:red", edgecolors="k", zorder=5)
ax.scatter(ca_sol, we_sol, marker="*", s=350, c="gold", edgecolors="k", zorder=6)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(log_limits(ca_all, BOUNDS["C_out"])); ax.set_ylim(log_limits(we_all, BOUNDS["W_in"]))
ax.set(xlabel=PARAM_LABELS["C_out"], ylabel=PARAM_LABELS["W_in"],
       title=f"Swarm exploration, run {rep_idx + 1} ({len(ca_all):,} evaluations)")
ax.legend(handles=[
    Line2D([], [], ls="", marker="o", ms=4, c="tab:green", alpha=0.6, label="evaluated positions"),
    Line2D([], [], c="k", marker="o", mfc="white", ms=5, label="global-best trajectory"),
    Line2D([], [], ls="", marker="s", ms=7, mfc="tab:red", mec="k", label="initial global best"),
    Line2D([], [], ls="", marker="*", ms=15, mfc="gold", mec="k", label="solution"),
    Line2D([], [], ls="--", c="grey", label="search range"),
], fontsize=8, loc="best")
ax.grid(True, which="both", alpha=0.2)

# (b) regime / diameter map at the solution's viscosities and radius
ca_g = np.geomspace(ca_lo, ca_hi, MAP_RESOLUTION)
we_g = np.geomspace(we_lo, we_hi, MAP_RESOLUTION)
CA, WE = np.meshgrid(ca_g, we_g)
mu_out_s, mu_in_s, r_in_s = rep_full[4], rep_full[5], rep_full[6]
grid = np.column_stack([
    CA.ravel(), WE.ravel(),
    (SIGMA * CA / mu_out_s).ravel(), np.sqrt(SIGMA * WE / (RHO_IN * 2 * r_in_s)).ravel(),
    np.full(CA.size, mu_out_s), np.full(CA.size, mu_in_s), np.full(CA.size, r_in_s),
])
gp = ensemble_predict(grid)
cls_map = gp["class"].reshape(CA.shape)
pdr_map = gp["p_dr_mean"].reshape(CA.shape)
d_map = np.ma.masked_where(cls_map != DRIPPING_CLASS, gp["d_mean"].reshape(CA.shape) * 1e6)

ax = axes[1]
ax.pcolormesh(CA, WE, cls_map, cmap=ListedColormap(CLASS_LIGHT), vmin=-0.5, vmax=2.5, shading="auto")
handles = [Patch(fc=CLASS_LIGHT[c], ec="grey", label=REGIME_NAMES[c]) for c in range(len(CLASS_LIGHT))]
if d_map.count() > 0:
    levels = MaxNLocator(10).tick_values(d_map.min(), d_map.max())
    cs = ax.contour(CA, WE, d_map, levels=levels, colors="#1f4e79", linewidths=0.7)
    ax.clabel(cs, fmt="%g µm", fontsize=7)
    handles.append(Line2D([], [], c="#1f4e79", lw=0.7, label="predicted diameter"))
    if d_map.min() < TARGET_DIAMETER_UM < d_map.max():
        ax.contour(CA, WE, d_map, levels=[TARGET_DIAMETER_UM], colors="k", linewidths=2.2)
        handles.append(Line2D([], [], c="k", lw=2.2, label=f"target diameter ({TARGET_DIAMETER_UM:g} µm)"))
if pdr_map.min() < P_THRESHOLD < pdr_map.max():
    ax.contour(CA, WE, pdr_map, levels=[P_THRESHOLD], colors="k", linestyles=":", linewidths=1.2)
    handles.append(Line2D([], [], c="k", ls=":", label=f"mean P(dripping) = {P_THRESHOLD:g}"))
ax.scatter(ca_sol, we_sol, marker="*", s=350, c="gold", edgecolors="k", zorder=6)
handles.append(Line2D([], [], ls="", marker="*", ms=15, mfc="gold", mec="k", label="solution"))
ax.set_xscale("log"); ax.set_yscale("log")
ax.set(xlabel=PARAM_LABELS["C_out"], ylabel=PARAM_LABELS["W_in"],
       title=(f"Map at $\\mu_{{out}}$ = {mu_out_s * 1e3:.3g} mPa s, $\\mu_{{in}}$ = {mu_in_s * 1e3:.3g} mPa s, "
              f"$r_{{in}}$ = {r_in_s * 1e6:.3g} µm"))
ax.legend(handles=handles, fontsize=8, loc="best", framealpha=0.9)
plt.tight_layout(); plt.show()

### 10.5 Ensemble predictions at the solution

**Left:** diameter predicted by each member of the diameter ensemble, with the ensemble mean, its confidence interval (shaded band) and the target. **Right:** class probabilities predicted by each member of the regime ensemble (dripping at the bottom of each bar), with the threshold $p_{th}$ used in the fitness term $t_1$. A design that is safely inside the dripping regime shows bars that are almost entirely blue and well above the threshold.

In [ ]:
#@title Report: per-member predictions
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

ax = axes[0]
idx_d = np.arange(1, n_d + 1)
ax.axhspan(d_mean - d_ci, d_mean + d_ci, color=CLASS_DARK[0], alpha=0.18, label=f"{LEVEL} CI of the mean")
ax.axhline(d_mean, c=CLASS_DARK[0], lw=1.5, label=f"ensemble mean = {d_mean:.2f} µm")
ax.axhline(TARGET_DIAMETER_UM, c="k", ls="--", lw=1.2, label=f"target = {TARGET_DIAMETER_UM:g} µm")
ax.scatter(idx_d, diams_m, c=CLASS_DARK[0], edgecolors="k", zorder=3, label="members")
ax.set(xlabel="member of the diameter ensemble", ylabel="predicted diameter [µm]",
       title="Diameter prediction at the solution", xticks=idx_d)
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)

ax = axes[1]
idx_r = np.arange(1, n_r + 1)
order = [DRIPPING_CLASS] + [c for c in range(probs_m.shape[1]) if c != DRIPPING_CLASS]
bottom = np.zeros(n_r)
for c in order:
    ax.bar(idx_r, probs_m[:, c], bottom=bottom, color=CLASS_DARK[c], label=REGIME_NAMES[c], width=0.8)
    bottom += probs_m[:, c]
ax.axhline(P_THRESHOLD, c="k", ls="--", lw=1.2, label=f"$p_{{th}}$ = {P_THRESHOLD:g}")
ax.set(xlabel="member of the regime ensemble", ylabel="class probability", ylim=(0, 1), xticks=idx_r,
       title=f"Regime prediction at the solution: {votes_dr}/{n_r} votes for dripping")
ax.legend(fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.17))
plt.tight_layout(); plt.show()

### 10.6 Comparison of the independent runs

**Left:** convergence of the global-best fitness in every run (the reported run is drawn thicker). **Centre:** contribution of each fitness term to the fitness of the reported solution. **Right:** the solution of every run, with each parameter shown as its position within its range on a logarithmic scale (0 = lower bound, 1 = upper bound); fixed parameters are shaded. Runs that failed the acceptance criteria of Section 9 are drawn with dotted lines. Runs that reach similar fitness with different parameters reveal alternative designs for the same target.

In [ ]:
#@title Report: comparison of runs
SHORT_LABELS = {"C_out": r"$Ca_{out}$", "W_in": r"$We_{in}$", "u_out": r"$u_{out}$", "u_in": r"$u_{in}$",
                "mu_out": r"$\mu_{out}$", "mu_in": r"$\mu_{in}$", "radius_in": r"$r_{in}$"}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), gridspec_kw={"width_ratios": [1.1, 1, 1.3]})

ax = axes[0]
for i, r in enumerate(run_results):
    rep = i == rep_idx
    ax.semilogy(r.history["f_best"], lw=2.4 if rep else 1, alpha=1 if rep else 0.7,
                ls="-" if i in accepted else ":",
                label=f"run {i + 1}: F = {r.f_best:.4f}" + (" (reported)" if rep else "")
                      + ("" if i in accepted else " (failed)"))
ax.set(xlabel="iteration", ylabel="global-best fitness F", title="Convergence of all runs")
ax.legend(fontsize=8); ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
y = np.arange(len(TERM_NAMES))[::-1]
ax.barh(y, rep_res.terms_best, color="tab:gray")
for yi, v in zip(y, rep_res.terms_best):
    ax.text(v + 0.01, yi, f"{v:.3f}", va="center", fontsize=8)
ax.set_yticks(y, TERM_NAMES); ax.set_xlim(0, 1.1)
ax.set(xlabel="term value", title=f"Fitness terms of the solution (F = {rep_res.f_best:.4f})")

ax = axes[2]
x = np.arange(len(PARAM_NAMES))
for j, name in enumerate(PARAM_NAMES):
    if name in SEARCH_VARS and FIXED[name] is not None:
        ax.axvspan(j - 0.4, j + 0.4, color="#ffe0b2", zorder=0)
for i, r in enumerate(run_results):
    f = to_full_inputs(r.x_best)[0]
    pos = [(np.log(f[j]) - np.log(BOUNDS[n][0])) / (np.log(BOUNDS[n][1]) - np.log(BOUNDS[n][0]))
           for j, n in enumerate(PARAM_NAMES)]
    rep = i == rep_idx
    ax.plot(x, pos, "-o" if i in accepted else ":o", lw=2.4 if rep else 1, ms=6 if rep else 4,
            alpha=1 if rep else 0.7,
            label=f"run {i + 1}" + (" (reported)" if rep else "") + ("" if i in accepted else " (failed)"))
ax.set_xticks(x, [SHORT_LABELS[n] for n in PARAM_NAMES])
ax.set(ylim=(-0.05, 1.05), ylabel="position within range (log scale)", title="Solutions of all runs")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Remarks

* The fitness uses **ensemble uncertainty** as a proxy for surrogate reliability, but members that share a common bias are not penalized by $t_3$. The paper reports that the surrogates tend to slightly overestimate the diameter (Section 5.3).
* The boundary penalty $t_4$ acts only on the first two search variables. Optima may therefore lie on the bounds of $\mu_{out}$, $\mu_{in}$ or $r_{in}$. If such a design is impractical, narrow those bounds or fix the parameter in `FIXED`.
* Every candidate design should be confirmed by a CFD simulation (as in Section 5.3 of the paper) or by experiment.